In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import pandas as pd
from IPython.display import display, Markdown

from emu_renewal.constants import OUTPUTS_PATH, ANALYSIS_NAMES
from emu_renewal.utils import get_country_short_name
from emu_renewal.outputs import get_idatas_for_mob_type
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios
from emu_renewal.plotting import plot_mob_exp_violins

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "46693102"
all_countries = ls(job_path)

# Purpose
This document presents estimates of 
the mobility exponent parameter posteriors by country and mobility analysis type.
Values from zero to two were considered plausible 
(the prior for this parameter was uniform over domain $[0, 2]$). 
A value of zero represents no effect, 
a value of one represents a linear scaling in transmission with mobility changes,
and a value of two represents a quadratic scaling in transmission with mobility changes 
(which could be epidemiologically considered as an effect on both infector and infectee).
For all violin plots, the vertical axis represents the mobility exponent parameter distribution,
while the darkness of the coloured shading represents the median of the relative
values of the dispersion parameter posterior under the analysis with mobility included
compared against the baseline simulation.

# Mobility exponent parameter posterior distributions by country

In [ ]:
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)
analyses = {k: v for k, v in ANALYSIS_NAMES.items() if "no_mob" not in k}

for mob_source, mob_name in analyses.items():

    # Median ratios
    ratio_vals = get_median_ratios(ratio_dists, mob_source)
    ratios = {get_country_short_name(k): v for k, v in ratio_vals.items()}
    
    # Mobility exponent distributions
    idatas, _ = get_idatas_for_mob_type(job_path, all_countries, mob_source)
    mob_exp_df = pd.DataFrame({c: idatas[c].posterior["mob_exp"].to_series() for c in idatas})

    # Plot
    display(Markdown(f"## {mob_name}"))
    display(plot_mob_exp_violins(mob_source, mob_exp_df, ratios))